# GPU 虚拟化全景：MIG、MPS、Time-slicing 与 vGPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-10-gpu-virtualization.ipynb)

把四种 GPU 共享/切分方式放进同一张图。本章以阅读和对照为主，用前九个 Lab 的直觉理解 MIG 硬隔离、MPS 软并发、time-slicing 时间片、vGPU 虚拟化各自的边界、代价与适用场景，并看清 Kubernetes 上各共享方案落在哪一类。

本 notebook 配合教程章节 [GPU 虚拟化全景：MIG、MPS、Time-slicing 与 vGPU](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

“GPU 共享/虚拟化”是个被滥用的词，背后其实是四种截然不同的机制。很多人把 MIG、vGPU、MPS 和 HAMi、KAI 这类 Kubernetes 共享方案混为一谈。本章把它们放进同一张比较图，让你在进入 Kubernetes 之前就建立清晰的边界，这样后面看到任何一个共享方案时，都能立刻知道它属于哪一类、补的是什么。

## 四种机制全景

![一张物理 GPU 的四种切法：MIG 硬件切分、MPS 软件并发、Time-slicing 时间片轮转、vGPU 厂商分片](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/four-sharing-mechanisms.svg)

## 横向对照表



| 维度 | MIG | MPS | Time-slicing | vGPU |
| :--- | :--- | :--- | :--- | :--- |
| 切的是什么 | 物理的 SM / 显存 / L2 / 显存控制器分区 | 软件层的 kernel 并发 | 驱动按时间片轮流独占 | Hypervisor 层把一张卡虚拟成多个虚拟 GPU 给虚拟机 |
| 隔离强度 | **最强**（硬件级，故障与性能都隔离） | 弱（客户端经同一个 server 提交，server 故障影响全部，性能互相影响） | 进程级（context 独立，故障隔离；但性能和显存不隔离） | 强（每个 vGPU 显存固定，算力按调度策略分配） |
| 配额粒度 | 固定几档 profile（如 A100 的 1g.5gb … 7g.40gb） | 环境变量，启动时定 | 无；Kubernetes device plugin 只能按“副本数”超卖 | 厂商 profile（如 T4-4Q = 4 GB） |
| T4 是否支持 | 不支持（需要 Ampere 及更新架构的数据中心卡） | 支持 | 支持 | 硬件支持，但需要 NVIDIA vGPU 软件许可与 hypervisor，**Colab 里用不到** |
| 谁来做 | 驱动 + NVML（`nvidia-smi mig`） | 驱动自带的 MPS daemon | 驱动默认行为 | NVIDIA vGPU 软件栈 / 国产卡厂商栈 |
| Kubernetes 上怎么用 | GPU Operator 的 MIG Manager，或 DRA 驱动 | device plugin 的 MPS 模式 | device plugin 按副本超卖；HAMi、KAI GPU 分数在其上加配额 | 虚拟机里当作一张较小的整卡使用 |

## 为什么 T4 支持不全反而是好教材

In [ ]:
!pip install -q nvidia-ml-py
from pynvml import *
nvmlInit()
h = nvmlDeviceGetHandleByIndex(0)
try:
    print("MIG:", nvmlDeviceGetMigMode(h))
except NVMLError_NotSupported:
    print("MIG on T4: not supported")
try:
    print("vGPU host mode:", nvmlDeviceGetVirtualizationMode(h))   # 0 = NONE，1 = PASSTHROUGH，3 = HOST_VGPU，4 = HOST_VSGA
except NVMLError as e:
    print("virtualization mode:", e)
print("compute mode  :", nvmlDeviceGetComputeMode(h))   # 0 = Default，允许多进程

实测输出：

```text
MIG on T4: not supported
vGPU host mode: 1
compute mode  : 0
```

三行分别回答三个问题：

- **能不能硬切？** 不能。T4 没有 MIG 功能，调用直接报“不支持”；
- **这张卡是怎么交给这台机器的？** `1` 表示 PASSTHROUGH（直通）：整张物理卡原样交给 Colab 背后的虚拟机，没有经过 vGPU 切分；
- **允许几个进程同时用？** `0` 表示 Default 模式，多个进程可以同时建 context。[Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 能同时跑两个 worker 就靠它；如果改成 `2`（Exclusive Process），第二个进程会直接报错。

T4 报“不支持 MIG”，这正好说明：**不是所有 GPU 都能做硬隔离**。真实数据中心里，A100 / A30 / H100 / H200 / B200 这些卡才能 MIG，而大量在跑推理的 T4 / L4 / L40S 不能，这就逼出了 MPS 和 time-slicing 这类软件方案。**HAMi、KAI GPU 分数这类软件配额方案，主要就是给那些“不能 MIG、但又要共享”的卡提供配额治理**。

`vGPU host mode` 返回 `1`（PASSTHROUGH）说明你拿到的是一张直通的整卡，而不是 vGPU 切出来的一片。Colab 的 GCE 实例就是这样：GPU 通过 PCIe passthrough 整张交给虚拟机，这是公有云 GPU 实例的主流做法。如果在 vGPU 虚拟机里查询，看到的会是 `2`（VGPU）。

## 三条决策线索

当你面对“这张卡要给多人用”时，按下面顺序判断：

1. **能不能 MIG？**（Ampere 及更新架构的数据中心卡，且业务要强隔离）→ 用 MIG，最干净，代价是档位固定、切换 profile 要清空负载。
2. **能不能接受软共享、但要有配额？** → 用软件配额方案，例如 HAMi（CUDA API 拦截）或 KAI GPU 分数（调度器记账，CUDA 13.4 起可由驱动强制显存上限），代价是性能隔离是软的。
3. **是虚拟机场景、或其他厂商的卡？** → 走 vGPU 或厂商方案（见 [多厂商设备](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/hands-on/multi-vendor-devices/)）。

![GPU 共享决策树：能 MIG 则用 MIG；否则要配额治理用软件配额方案；否则虚拟机或厂商专用卡走 vGPU 或厂商方案](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/sharing-decision-tree.svg)

## 验证

完成后你应该能：

- 说出四种机制的切分对象与隔离强度；
- 解释 T4 为何不支持 MIG、这为什么反而重要；
- 解释“T4 硬件支持 vGPU”和“Colab 里用不到 vGPU”为什么不矛盾；
- 把 HAMi、KAI GPU 分数定位到“time-slicing 之上的软共享 + 配额治理”这一格；
- 用决策图判断一个新场景该用哪种机制。

## 故障模式



| 困惑 | 解答 |
| :--- | :--- |
| “软件配额方案是不是就是 MIG？” | 不是。MIG 是硬件切分；HAMi、KAI GPU 分数是 Kubernetes 之上的配额调度，有的可以对接 MIG，但不等于它。 |
| “vGPU 和 Kubernetes 共享方案冲突吗？” | 不冲突，vGPU 在 hypervisor 层，共享方案在 Kubernetes 层；虚拟机里的 vGPU 对它们来说就是一张“小一点的卡”。其他厂商的卡见 [多厂商设备](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/hands-on/multi-vendor-devices/)。 |
| “MPS 就是软件配额吗？” | 不是。[Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 已说明 MPS 只解决并发，HAMi-core 这类方案走的是 CUDA API 拦截。 |
| “Time-slicing 不是已经有隔离了吗？” | 它只有 context 级的故障隔离，没有显存上限和算力配额，这正是软件配额方案补的部分。 |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。